## ⚙️ Load the GROMACS–PLUMED Environment

Loads the required **compiler, MPI, CUDA, GROMACS, and PLUMED environment** on ACCRE and makes the configured GROMACS–PLUMED installation available within the notebook for subsequent HREMD setup and simulations.

In [2]:
import os
import subprocess

def load_gmx_plumed():
    cmd = """
    module load StdEnvACCRE/2023
    module load gcc/12.3
    module load openmpi/4.1.5
    module load cuda/12.2
    source $HOME/gromacs-2023.5-plumed-mpi/bin/GMXRC
    export PLUMED_KERNEL=$HOME/plumed-2.9.2/lib/libplumedKernel.so
    export LD_LIBRARY_PATH=$HOME/plumed-2.9.2/lib:$LD_LIBRARY_PATH
    export PATH=$HOME/plumed-2.9.2/bin:$PATH
    env
    """
    # run in bash, capture environment
    result = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    for line in result.stdout.splitlines():
        key, _, value = line.partition("=")
        os.environ[key] = value


In [3]:
load_gmx_plumed()

## 🧪 Test the 64-Replica Setup Before SLURM Submission

Runs all **64 λ replicas simultaneously** as a test to confirm that the multi-replica setup launches and runs correctly **before submitting the full HREMD SLURM job**.

In [ ]:
%%bash
cd Hamiltonian 
mpirun --oversubscribe -np 64 gmx_mpi mdrun -multidir lambda*/ -nsteps 10000000 -nb gpu -pme gpu -bonded gpu -pin auto -ntomp 1

bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

## 🚀 Equilibrate the 64 Replicas and Start Production HREMD

Submits the **64-replica HREMD simulation** on ACCRE using four GPUs.

- Runs all **64 λ replicas simultaneously**.
- Performs **20 ns of equilibration** for each replica before replica exchange.
- Following equilibration, starts the **production HREMD simulation** across all 64 replicas.
- Attempts **Hamiltonian exchanges every 1,000 MD steps**.
- Runs production for **250 million steps (500 ns at a 2 fs timestep)** per replica.

In [ ]:
%%bash

cat > Hamiltonian/3PRG_Hamiltonian.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=3PRG
#SBATCH --mail-user=mithun.nag.karadi.giridhar@vanderbilt.edu
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=64
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --time=10-00:00:00
#SBATCH --gres=gpu:nvidia_rtx_a6000:4
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=3PRG_out_%j.log
#SBATCH --error=3PRG_err_%j.log

# === Environment ===
module load StdEnvACCRE/2023
module load gcc/12.3 openmpi/4.1.5 cuda/12.2
source $HOME/gromacs-2023.5-plumed-mpi/bin/GMXRC
export PLUMED_KERNEL=$HOME/plumed-2.9.2/lib/libplumedKernel.so
export LD_LIBRARY_PATH=$HOME/plumed-2.9.2/lib:$LD_LIBRARY_PATH
export PATH=$HOME/plumed-2.9.2/bin:$PATH
export OMP_NUM_THREADS=1
export GMX_DISABLE_GPU_TIMING=1
export GMX_ENABLE_DIRECT_GPU_COMM=1
export SLURM_CPU_BIND=none
unset CUDA_VISIBLE_DEVICES

# === Move into Hamiltonian ===
cd Hamiltonian || exit 1
echo "PWD: $(pwd)"

# === Collect replica directories into an array ===
dirs=(lambda*/)
echo "Found ${#dirs[@]} replicas:"
printf '%s\n' "${dirs[@]}"

# === Equilibration (20 ns ≈ 10 million steps) ===
mpirun -np 64 gmx_mpi mdrun \
    -multidir "${dirs[@]}" \
    -nsteps 10000000 \
    -nb gpu -pme gpu -bonded gpu \
    -pin auto -ntomp 1


echo "✅ Equilibration complete — check md_eq.cpt"

# === Production H-REMD ===
mpirun -np 64 gmx_mpi mdrun \
    -multidir "${dirs[@]}" \
    -cpi state.cpt -noappend \
    -nsteps 250000000 -hrex -replex 1000 -plumed plumed.dat \
    -nb gpu -pme gpu -bonded gpu \
    -pin auto -ntomp 1 \
    -deffnm md_prod


echo "🎯 H-REMD started from md_eq.cpt — job done $(date)"
EOF

# submit the job
sbatch Hamiltonian/3PRG_Hamiltonian.slurm


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

Submitted batch job 11465934


## 🚀 Restart Production HREMD with Replica Exchange

The initial **20 ns equilibration without replica exchange was completed successfully**, but the subsequent production HREMD run did not complete. This step therefore **restarts the production simulation from the existing checkpoints**, with Hamiltonian replica exchange enabled (`-hrex`) and exchange attempts every **1,000 MD steps**, continuing the 64-replica HREMD production run.

In [ ]:
%%bash

cat > Hamiltonian/3PRG_Hamiltonian.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=3PRG
#SBATCH --mail-user=mithun.nag.karadi.giridhar@vanderbilt.edu
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=64
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --time=10-00:00:00
#SBATCH --gres=gpu:nvidia_rtx_a6000:4
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=3PRG_out_%j.log
#SBATCH --error=3PRG_err_%j.log

# === Environment ===
module load StdEnvACCRE/2023
module load gcc/12.3 openmpi/4.1.5 cuda/12.2
source $HOME/gromacs-2023.5-plumed-mpi/bin/GMXRC
export PLUMED_KERNEL=$HOME/plumed-2.9.2/lib/libplumedKernel.so
export LD_LIBRARY_PATH=$HOME/plumed-2.9.2/lib:$LD_LIBRARY_PATH
export PATH=$HOME/plumed-2.9.2/bin:$PATH
export OMP_NUM_THREADS=1
export GMX_DISABLE_GPU_TIMING=1
export GMX_ENABLE_DIRECT_GPU_COMM=1
export SLURM_CPU_BIND=none
unset CUDA_VISIBLE_DEVICES

# === Move into Hamiltonian ===
cd Hamiltonian || exit 1
echo "PWD: $(pwd)"

# === Collect replica directories into an array ===
dirs=(lambda*/)
echo "Found ${#dirs[@]} replicas:"
printf '%s\n' "${dirs[@]}"


# === Production H-REMD ===
mpirun -np 64 gmx_mpi mdrun \
    -multidir "${dirs[@]}" \
    -cpi state.cpt -noappend \
    -nsteps 250000000 -hrex -replex 1000 -plumed plumed.dat \
    -nb gpu -pme gpu -bonded gpu \
    -pin auto -ntomp 1 \
    -deffnm md_prod


echo "🎯 H-REMD started from md_eq.cpt — job done $(date)"
EOF

# submit the job
sbatch Hamiltonian/3PRG_Hamiltonian.slurm


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

Submitted batch job 11465935
